In [1]:
import os
import regex as re
import unicodedata
import numpy as np
import pandas as pd
import math
from collections import Counter

# For modeling and balancing:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.ensemble import RandomForestClassifier  

#############################################
# 1. DATA CLEANING
#############################################
def clean_text(text):
    """
    Basic text cleaning:
    - Converts input to string
    - Removes numbers and punctuation
    - Removes multiple spaces
    - Converts to lowercase and normalizes Unicode
    """
    text = str(text)
    text = re.sub(r'\d', '', text)  # Remove digits
    text = re.sub(r'[^\w\s]', '', text)  # Remove punctuation
    text = re.sub(r'\s+', ' ', text).strip()  # Normalize spaces
    text = text.lower()
    text = unicodedata.normalize('NFC', text)
    return text

#############################################
# 2. FEATURE ENGINEERING
#############################################
# 2.1 Extract script information
import regex as re2  # 'regex' supports advanced Unicode handling

def detect_script(text):
    """
    Detects the most frequent script used in the given text.
    """
    scripts = {}
    for char in text:
        try:
            script = unicodedata.name(char).split()[0]  # Extract script name
            scripts[script] = scripts.get(script, 0) + 1
        except ValueError:
            continue  # Ignore characters without a name
    return max(scripts, key=scripts.get, default="Unknown")  # Handle empty cases

# 2.2 Compute character entropy (considering only letters)
def char_entropy(text):
    """
    Computes the entropy of characters in a given text,
    considering only letters.
    """
    letters = re.findall(r'\p{L}', text)  # Extract only letters using regex
    if not letters:
        return 0.0
    counts = Counter(letters)
    total = sum(counts.values())
    entropy = -sum((count / total) * math.log2(count / total) for count in counts.values())
    return entropy

#############################################
# DATA LOADING AND INITIAL PREPROCESSING
#############################################
DATA_PATH = "../train_submission.csv" 
df = pd.read_csv(DATA_PATH, encoding='utf-8')

# Apply basic text cleaning
df['Text_clean'] = df['Text'].apply(clean_text)

# Add script feature
df['Script'] = df['Text_clean'].apply(detect_script)

In [30]:
df.head()

,Usage,Text,Label,Text_clean,Script
0,Public,َ قَالَ النَّبِيُّ ص إِنِّي أَتَعَجَّبُ مِمَّن...,hau,َ قَالَ النَّبِيُّ ص إِنِّي أَتَعَجَّبُ مِمَّن...,ARABIC
1,Public,Filmen forteller historien om Will Hunting en...,nob,filmen forteller historien om will hunting en ...,LATIN
2,Public,An Arthrostylidium berryi in uska species han ...,wln,an arthrostylidium berryi in uska species han ...,LATIN
3,Public,Kancunarí enemigosniyquichejta munacuychej al...,quh,kancunarí enemigosniyquichejta munacuychej all...,LATIN
4,Public,Warmeqa ama yachachichunchu hermanospa tantaku...,quh,warmeqa ama yachachichunchu hermanospa tantaku...,LATIN


In [ ]:
# 1. Add entropy
df['char_entropy'] = df['Text_clean'].apply(char_entropy)

# Visualize the first few rows to check
print(df.head())

#############################################
# 2. REMOVE DATA WITH FEW EXAMPLES
#############################################
# Define the threshold (e.g., minimum 50 examples)
min_examples = 50
label_counts = df['Label'].value_counts()
labels_to_keep = label_counts[label_counts >= min_examples].index
df_filtered = df[df['Label'].isin(labels_to_keep)].copy()
print("Number of examples after filtering:", df_filtered.shape[0])

#############################################
# 3. FEATURE PREPARATION AND BALANCING
#############################################
# We will use TF-IDF for the textual part and keep additional features
# Function to extract only the text column
def extract_text(df):
    return df['Text_clean']

# Function to extract additional numeric features
def extract_numeric_features(df):
    # Assuming the columns for script and entropy are as follows:
    num_cols = [col for col in df.columns if col.startswith('prop_')]
    num_cols.append('char_entropy')
    return df[num_cols]

# Create transformers
text_transformer = Pipeline([
    ('selector', FunctionTransformer(lambda x: x, validate=False)),  # receives the already selected column
    ('tfidf', TfidfVectorizer(max_features=10000))
])

numeric_transformer = Pipeline([
    ('selector', FunctionTransformer(lambda df: df, validate=False)),
    ('scaler', StandardScaler())
])

# Use FeatureUnion to combine the two parts.
# We need to create a DataFrame with both parts: one for text and another for numeric features.
# Alternatively, we can use a custom approach. Here, we'll concatenate the results.
def combined_features(df):
    text_feats = text_transformer.fit_transform(df['Text_clean'])
    numeric_feats = numeric_transformer.fit_transform(extract_numeric_features(df))
    # If text_feats is sparse, convert numeric_feats to sparse as well and hstack
    from scipy.sparse import hstack, csr_matrix
    numeric_feats_sparse = csr_matrix(numeric_feats)
    return hstack([text_feats, numeric_feats_sparse])


    Usage                                               Text Label  \
0  Public  َ قَالَ النَّبِيُّ ص إِنِّي أَتَعَجَّبُ مِمَّن...   hau   
1  Public  Filmen forteller historien om Will Hunting  en...   nob   
2  Public  An Arthrostylidium berryi in uska species han ...   wln   
3  Public  Kancunarí enemigosniyquichejta munacuychej  al...   quh   
4  Public  Warmeqa ama yachachichunchu hermanospa tantaku...   quh   

                                          Text_clean  Script  char_entropy  
0  َ قَالَ النَّبِيُّ ص إِنِّي أَتَعَجَّبُ مِمَّن...  ARABIC      4.498605  
1  filmen forteller historien om will hunting en ...   LATIN      4.074268  
2  an arthrostylidium berryi in uska species han ...   LATIN      4.086681  
3  kancunarí enemigosniyquichejta munacuychej all...   LATIN      3.985452  
4  warmeqa ama yachachichunchu hermanospa tantaku...   LATIN      3.797551  
Número de exemplos após filtragem: 189854


In [ ]:
# First, create the combined features for the entire dataset
X = combined_features(df_filtered)
y = df_filtered['Label'].values

#############################################
# 4.1 BALANCING: UNDERSAMPLING AND OVERSAMPLING
#############################################
# We will use a combination strategy.
# For example, we can apply SMOTE for oversampling and RandomUnderSampler for undersampling.
from imblearn.combine import SMOTETomek

# SMOTETomek combines SMOTE (oversampling) with Tomek links (undersampling) to clean the class boundary.
smt = SMOTETomek(random_state=42)
X_res, y_res = smt.fit_resample(X, y)

# print("Class distribution after resampling:", np.unique(y_res, return_counts=True))


Distribuição das classes após reamostragem: (array(['abk', 'ace', 'ach', 'acm', 'acr', 'afb', 'afr', 'ahk', 'ajp',
       'aka', 'aln', 'als', 'alt', 'amh', 'aoj', 'apc', 'ara', 'arb',
       'arg', 'arn', 'ary', 'arz', 'asm', 'ast', 'aym', 'ayr', 'azb',
       'aze', 'azj', 'bak', 'bam', 'ban', 'bar', 'bcl', 'bel', 'bem',
       'ber', 'bew', 'bih', 'bik', 'bis', 'bjn', 'bod', 'bos', 'bpy',
       'bqc', 'bre', 'bsb', 'bul', 'bzj', 'cab', 'cak', 'cat', 'cbk',
       'ceb', 'ces', 'che', 'chk', 'chv', 'cjk', 'ckb', 'cmn', 'cos',
       'crh', 'csb', 'csy', 'ctu', 'cuk', 'cym', 'dan', 'deu', 'diq',
       'div', 'djk', 'dtp', 'dyu', 'dzo', 'ekk', 'ell', 'eml', 'eng',
       'epo', 'est', 'eus', 'ewe', 'ext', 'fao', 'fas', 'fij', 'fil',
       'fin', 'fon', 'fra', 'frr', 'fry', 'ful', 'fur', 'gcf', 'gla',
       'gle', 'glg', 'glk', 'glv', 'gom', 'gor', 'grc', 'grn', 'gsw',
       'guc', 'gug', 'guj', 'gym', 'hat', 'hau', 'haw', 'hbs', 'heb',
       'hif', 'hil', 'hin', 'hmn', 'hmo', 'hn

In [ ]:
from scipy.sparse import save_npz
import numpy as np

# Assume your CSR matrix is 'X_res'
save_npz("X_res.npz", X_res)  # Save the CSR matrix to a .npz file
np.save("y_res.npy", y_res)   # Save the labels to a .npy file


In [ ]:
from sklearn.preprocessing import LabelEncoder

# Create the encoder and fit it to the unique labels
label_encoder = LabelEncoder()
y_res_encoded = label_encoder.fit_transform(y_res)

In [ ]:
from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
import numpy as np

# Calculate class weights (balanced)
classes = np.unique(y_res_encoded)
class_weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_res_encoded)
class_weight_dict = {cls: weight for cls, weight in zip(classes, class_weights)}

# Assign weights to the samples
sample_weight = np.array([class_weight_dict[label] for label in y_res_encoded])

# XGBoost model adjusted for imbalanced data
xgb_clf = XGBClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=6,
    random_state=42,
    n_jobs=-1,
    tree_method='hist'
)

# Create stratified cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = []

# Manual loop for proper model validation
for train_idx, test_idx in cv.split(X_res, y_res_encoded):
    X_train, X_test = X_res[train_idx], X_res[test_idx]
    y_train, y_test = y_res_encoded[train_idx], y_res_encoded[test_idx]
    
    # Apply sample weights only to the training set
    sample_weight_train = sample_weight[train_idx]

    # Train the model
    xgb_clf.fit(X_train, y_train, sample_weight=sample_weight_train)
    
    # Make predictions
    y_pred = xgb_clf.predict(X_test)
    
    # Evaluate the model with Macro F1 score
    f1 = f1_score(y_test, y_pred, average='macro')
    scores.append(f1)

# Output the average and standard deviation of the Macro F1 score
print(f"Macro F1 Score (manual cross-validation): {np.mean(scores):.4f} ± {np.std(scores):.4f}")


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
import numpy as np

# Calculate class weights (balanced)
classes = np.unique(y_res_encoded)
class_weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_res_encoded)
class_weight_dict = {cls: weight for cls, weight in zip(classes, class_weights)}

# Assign weights to the samples
sample_weight = np.array([class_weight_dict[label] for label in y_res_encoded])

# RandomForest model adjusted for imbalanced data
rf_clf = RandomForestClassifier(
    n_estimators=100, random_state=42
)

# Create stratified cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = []

# Manual loop for proper model validation
for train_idx, test_idx in cv.split(X_res, y_res_encoded):
    X_train, X_test = X_res[train_idx], X_res[test_idx]
    y_train, y_test = y_res_encoded[train_idx], y_res_encoded[test_idx]
    
    # Apply sample weights only to the training set
    sample_weight_train = sample_weight[train_idx]

    # Train the model
    rf_clf.fit(X_train, y_train, sample_weight=sample_weight_train)
    
    # Make predictions
    y_pred = rf_clf.predict(X_test)
    
    # Evaluate the model with Macro F1 score
    f1 = f1_score(y_test, y_pred, average='macro')
    scores.append(f1)

# Output the average and standard deviation of the Macro F1 score
print(f"Macro F1 Score (manual cross-validation): {np.mean(scores):.4f} ± {np.std(scores):.4f}")